In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

from pathlib import Path

ROOT = Path.cwd().parent
print(ROOT)

DATA_RAW = ROOT/"data/raw"
DATA_PROCESSED = ROOT/"data/processed"

c:\Users\sebas\PycharmProjects\Git\BoxOffice_Oracle


In [4]:
model_df = pd.read_csv(
    DATA_RAW/"00_model_df.csv"
)

print(model_df.shape)
model_df.head()

(2255, 42)


,tconst,primaryTitle,startYear,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,all_domestic_release_types,distributor,production_budget,...,actor_4,actor_5,actor_6,director_name,writer_name,actor_1_name,actor_2_name,actor_3_name,franchise,final_market_synopsis
0,tt0120667,Fantastic Four,2005.0,56061504.0,3602.0,2005-07-08,Wide,Wide,20th Century Fox,87500000.0,...,nm0004695,nm0573037,nm0512934,Tim Story,Mark Frost,Ioan Gruffudd,Michael Chiklis,Chris Evans,Fantastic Four,Scientist Reed Richards persuades his arrogant...
1,tt0121164,Corpse Bride,2005.0,19145480.0,3204.0,2005-09-16,Expands Wide,Expands Wide,Warner Bros.,40000000.0,...,nm0001808,nm0925768,NaN,Tim Burton,John August,Johnny Depp,Helena Bonham Carter,Emily Watson,NaN,"In a 19th-century European village, a young ma..."
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,108435841.0,3661.0,2005-05-19,Wide,Wide,20th Century Fox,115000000.0,...,nm0000168,nm0001519,nm0001751,George Lucas,George Lucas,Hayden Christensen,Natalie Portman,Ewan McGregor,Star Wars,"Years after the onset of the Clone Wars, the n..."
3,tt0167190,Hellboy,2004.0,23172440.0,3028.0,2004-04-02,Wide,Wide,Sony Pictures,60000000.0,...,nm0000457,nm1140344,nm0734558,Guillermo del Toro,Guillermo del Toro,Ron Perlman,Doug Jones,Selma Blair,Hellboy,"In the final days of World War II, the Nazis a..."
4,tt0200465,The Bank Job,2008.0,5935256.0,1603.0,2008-03-07,Wide,Wide,Lionsgate,20000000.0,...,nm0990547,nm0269077,nm0202810,Roger Donaldson,Dick Clement,Jason Statham,Saffron Burrows,Stephen Campbell Moore,NaN,Self-reformed petty criminal Terry Leather has...


In [5]:
model_df.columns

Index(['tconst', 'primaryTitle', 'startYear', 'opening_weekend_gross',
       'opening_theaters', 'domestic_release_date', 'release_type',
       'all_domestic_release_types', 'distributor', 'production_budget',
       'MPA_rating', 'MPA_rating_reason', 'runtime_minutes', 'genre',
       'subgenre', 'target_audience', 'time_period_setting', 'source',
       'production_method', 'creative_type', 'production_countries',
       'languages', 'legs', 'plot_point', 'log_opening_weekend_gross',
       'release_month', 'release_day_of_year', 'director_id', 'writer_id',
       'actor_1', 'actor_2', 'actor_3', 'actor_4', 'actor_5', 'actor_6',
       'director_name', 'writer_name', 'actor_1_name', 'actor_2_name',
       'actor_3_name', 'franchise', 'final_market_synopsis'],
      dtype='str')

In [30]:
def add_group1_prior_genre_experience(model_df):
    df = model_df.copy()

    # Make sure dates/years sort correctly
    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")
    df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")

    # Stable chronological order
    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    # Entity columns to test
    entity_cols = {
        "director": "director_id",
        "writer": "writer_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "distributor": "distributor"
    }

    # Prior films by entity within same genre
    for label, col in entity_cols.items():
        feature_name = f"g1_prior_same_genre_{label}_count"

        df[feature_name] = (
            df.groupby([col, "genre"])
              .cumcount()
        )

        # If entity is missing, set to 0
        df.loc[df[col].isna(), feature_name] = 0

    # Combined actor feature: total prior same-genre experience across top 3 actors
    actor_features = [
        "g1_prior_same_genre_actor_1_count",
        "g1_prior_same_genre_actor_2_count",
        "g1_prior_same_genre_actor_3_count"
    ]

    df["g1_prior_same_genre_top3_actor_total"] = df[actor_features].sum(axis=1)
    df["g1_prior_same_genre_top3_actor_max"] = df[actor_features].max(axis=1)
    df["g1_prior_same_genre_top3_actor_mean"] = df[actor_features].mean(axis=1)

    return df

In [46]:
model_df_g1 = add_group1_prior_genre_experience(model_df)

g1_features = [col for col in model_df_g1.columns if col.startswith("g1_")]

g1_features

['g1_prior_same_genre_director_count',
 'g1_prior_same_genre_writer_count',
 'g1_prior_same_genre_actor_1_count',
 'g1_prior_same_genre_actor_2_count',
 'g1_prior_same_genre_actor_3_count',
 'g1_prior_same_genre_distributor_count',
 'g1_prior_same_genre_top3_actor_total',
 'g1_prior_same_genre_top3_actor_max',
 'g1_prior_same_genre_top3_actor_mean']

In [47]:
model_df_g1.to_csv(DATA_RAW/"fe_groups/g1.csv")

## Group 2
### 3. Historical Momentum Features
- Actor, director, writer, studio, distributor, or genre gross revenue acceleration.
- Possible versions:
  - revenue velocity
  - change in revenue velocity
  - rolling historical growth rate

In [31]:
def add_group2b_time_aware_momentum(model_df):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(
        df["domestic_release_date"],
        errors="coerce"
    )

    df["opening_weekend_gross"] = pd.to_numeric(
        df["opening_weekend_gross"],
        errors="coerce"
    )

    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    entity_cols = {
        "director": "director_id",
        "writer": "writer_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "distributor": "distributor",
        "genre": "genre"
    }

    for label, col in entity_cols.items():

        group = df.groupby(col, dropna=False)

        prev_gross_1 = group["opening_weekend_gross"].shift(1)
        prev_gross_2 = group["opening_weekend_gross"].shift(2)
        prev_gross_3 = group["opening_weekend_gross"].shift(3)

        prev_date_1 = group["domestic_release_date"].shift(1)
        prev_date_2 = group["domestic_release_date"].shift(2)
        prev_date_3 = group["domestic_release_date"].shift(3)

        # Your original intuition: recent raw movement
        df[f"g2b_raw_change_last_{label}"] = (
            prev_gross_1 - prev_gross_2
        )

        df[f"g2b_raw_acceleration_last_{label}"] = (
            (prev_gross_1 - prev_gross_2)
            -
            (prev_gross_2 - prev_gross_3)
        )

        # Better scale-aware version: percent movement
        df[f"g2b_pct_change_last_{label}"] = (
            (prev_gross_1 - prev_gross_2) / prev_gross_2.replace(0, np.nan)
        )

        df[f"g2b_pct_acceleration_last_{label}"] = (
            df[f"g2b_pct_change_last_{label}"]
            -
            ((prev_gross_2 - prev_gross_3) / prev_gross_3.replace(0, np.nan))
        )

        # Time-aware velocity: change per day between prior projects
        days_12 = (prev_date_1 - prev_date_2).dt.days
        days_23 = (prev_date_2 - prev_date_3).dt.days

        df[f"g2b_velocity_per_day_last_{label}"] = (
            (prev_gross_1 - prev_gross_2) / days_12.replace(0, np.nan)
        )

        previous_velocity = (
            (prev_gross_2 - prev_gross_3) / days_23.replace(0, np.nan)
        )

        # Time-aware acceleration: change in velocity
        # divided by time between velocity measurements
        midpoint_gap_days = (
            (prev_date_1 - prev_date_3).dt.days / 2
        )

        df[f"g2b_acceleration_per_day_last_{label}"] = (
            (
                df[f"g2b_velocity_per_day_last_{label}"]
                -
                previous_velocity
            )
            / midpoint_gap_days.replace(0, np.nan)
        )

        # Direction flags: easier for tree models to use
        df[f"g2b_is_heating_up_{label}"] = (
            df[f"g2b_pct_change_last_{label}"] > 0.10
        ).astype(int)

        df[f"g2b_is_cooling_down_{label}"] = (
            df[f"g2b_pct_change_last_{label}"] < -0.10
        ).astype(int)

        df[f"g2b_is_neutral_momentum_{label}"] = (
            df[f"g2b_pct_change_last_{label}"].between(-0.10, 0.10)
        ).astype(int)

        df.loc[df[col].isna(), [
            f"g2b_raw_change_last_{label}",
            f"g2b_raw_acceleration_last_{label}",
            f"g2b_pct_change_last_{label}",
            f"g2b_pct_acceleration_last_{label}",
            f"g2b_velocity_per_day_last_{label}",
            f"g2b_acceleration_per_day_last_{label}",
            f"g2b_is_heating_up_{label}",
            f"g2b_is_cooling_down_{label}",
            f"g2b_is_neutral_momentum_{label}",
        ]] = np.nan

    actor_cols = [
        "actor_1",
        "actor_2",
        "actor_3"
    ]

    for metric in [
        "pct_change_last",
        "pct_acceleration_last",
        "velocity_per_day_last",
        "acceleration_per_day_last",
    ]:
        cols = [
            f"g2b_{metric}_{actor}"
            for actor in actor_cols
        ]

        df[f"g2b_top3_actor_{metric}_mean"] = df[cols].mean(axis=1)
        df[f"g2b_top3_actor_{metric}_max"] = df[cols].max(axis=1)
        df[f"g2b_top3_actor_{metric}_min"] = df[cols].min(axis=1)

    return df

In [13]:
model_df_g2b = add_group2b_time_aware_momentum(model_df)

g2b_features = [
    col for col in model_df_g2b.columns
    if col.startswith("g2b_")
]

len(g2b_features), g2b_features[:20]

(75,
 ['g2b_raw_change_last_director',
  'g2b_raw_acceleration_last_director',
  'g2b_pct_change_last_director',
  'g2b_pct_acceleration_last_director',
  'g2b_velocity_per_day_last_director',
  'g2b_acceleration_per_day_last_director',
  'g2b_is_heating_up_director',
  'g2b_is_cooling_down_director',
  'g2b_is_neutral_momentum_director',
  'g2b_raw_change_last_writer',
  'g2b_raw_acceleration_last_writer',
  'g2b_pct_change_last_writer',
  'g2b_pct_acceleration_last_writer',
  'g2b_velocity_per_day_last_writer',
  'g2b_acceleration_per_day_last_writer',
  'g2b_is_heating_up_writer',
  'g2b_is_cooling_down_writer',
  'g2b_is_neutral_momentum_writer',
  'g2b_raw_change_last_actor_1',
  'g2b_raw_acceleration_last_actor_1'])

In [12]:
model_df_g2 = add_group2b_time_aware_momentum(model_df)

g2_features = [col for col in model_df_g2.columns if col.startswith("g2_")]
g2_features

[]

In [14]:
model_df_g2b.to_csv(DATA_RAW/"fe_groups/g2.csv")

## Group 3
### Release Timing and Competition Features
- Distance from nearest seasonal demand peak.
- Nearest seasonal demand peak category.
- Number of similar-genre movies released within 20 days.
- Number of general movies released within 20 days.
- Number of similar-genre movies with similar budgets released within 20 days.
- Number of general movies with similar budgets released within 20 days.
- Budget × distance from nearest seasonal peak interaction.

In [32]:
def add_group3_release_timing_competition(model_df, window_days=20, budget_tolerance=0.30):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")
    df["production_budget"] = pd.to_numeric(df["production_budget"], errors="coerce")
    df["release_day_of_year"] = pd.to_numeric(df["release_day_of_year"], errors="coerce")

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    # Approximate seasonal demand peaks
    seasonal_peaks = {
        "summer": 185,        # early July
        "thanksgiving": 330,  # late November
        "christmas": 355,     # late December
        "spring": 90,         # late March / early April
        "halloween": 300      # late October
    }

    def nearest_peak(day):
        if pd.isna(day):
            return pd.Series([np.nan, np.nan])

        distances = {
            peak: min(abs(day - peak_day), 365 - abs(day - peak_day))
            for peak, peak_day in seasonal_peaks.items()
        }

        nearest = min(distances, key=distances.get)
        return pd.Series([distances[nearest], nearest])

    df[["g3_distance_from_nearest_seasonal_peak", "g3_nearest_seasonal_peak_category"]] = (
        df["release_day_of_year"].apply(nearest_peak)
    )

    # Competition features
    general_counts = []
    similar_genre_counts = []
    general_similar_budget_counts = []
    similar_genre_similar_budget_counts = []

    for idx, row in df.iterrows():
        release_date = row["domestic_release_date"]
        genre = row["genre"]
        budget = row["production_budget"]

        if pd.isna(release_date):
            general_counts.append(np.nan)
            similar_genre_counts.append(np.nan)
            general_similar_budget_counts.append(np.nan)
            similar_genre_similar_budget_counts.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        general_counts.append(len(nearby))

        similar_genre = nearby[nearby["genre"] == genre]
        similar_genre_counts.append(len(similar_genre))

        if pd.isna(budget) or budget <= 0:
            general_similar_budget_counts.append(np.nan)
            similar_genre_similar_budget_counts.append(np.nan)
        else:
            lower_budget = budget * (1 - budget_tolerance)
            upper_budget = budget * (1 + budget_tolerance)

            nearby_similar_budget = nearby[
                (nearby["production_budget"] >= lower_budget) &
                (nearby["production_budget"] <= upper_budget)
            ]

            general_similar_budget_counts.append(len(nearby_similar_budget))

            similar_genre_similar_budget_counts.append(
                len(nearby_similar_budget[nearby_similar_budget["genre"] == genre])
            )

    df["g3_nearby_general_movies_20d_count"] = general_counts
    df["g3_nearby_similar_genre_movies_20d_count"] = similar_genre_counts
    df["g3_nearby_general_similar_budget_movies_20d_count"] = general_similar_budget_counts
    df["g3_nearby_similar_genre_similar_budget_movies_20d_count"] = similar_genre_similar_budget_counts

    # Interaction feature
    df["g3_budget_x_distance_from_peak"] = (
        df["production_budget"] * df["g3_distance_from_nearest_seasonal_peak"]
    )

    return df

In [ ]:
model_df_g3 = add_group3_release_timing_competition(model_df)

g3_features = [col for col in model_df_g3.columns if col.startswith("g3_")]
g3_features

['g3_distance_from_nearest_seasonal_peak',
 'g3_nearest_seasonal_peak_category',
 'g3_nearby_general_movies_20d_count',
 'g3_nearby_similar_genre_movies_20d_count',
 'g3_nearby_general_similar_budget_movies_20d_count',
 'g3_nearby_similar_genre_similar_budget_movies_20d_count',
 'g3_budget_x_distance_from_peak']

In [53]:
model_df_g3.to_csv(DATA_RAW/"fe_groups/g3.csv")

## Group 4
### Local Genre Environment
- Mode genre among movies released within 60 days of the target movie.


In [33]:
def add_group4_local_genre_environment(model_df, window_days=60):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    mode_genres = []
    mode_genre_counts = []

    for idx, row in df.iterrows():
        release_date = row["domestic_release_date"]

        if pd.isna(release_date):
            mode_genres.append(np.nan)
            mode_genre_counts.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        if nearby.empty or nearby["genre"].dropna().empty:
            mode_genres.append(np.nan)
            mode_genre_counts.append(0)
        else:
            counts = nearby["genre"].value_counts()
            mode_genres.append(counts.index[0])
            mode_genre_counts.append(counts.iloc[0])

    df["g4_local_60d_mode_genre"] = mode_genres
    df["g4_local_60d_mode_genre_count"] = mode_genre_counts

    # Whether this movie matches the local dominant genre
    df["g4_matches_local_60d_mode_genre"] = (
        df["genre"] == df["g4_local_60d_mode_genre"]
    ).astype(int)

    return df

In [55]:
model_df_g4 = add_group4_local_genre_environment(model_df)

g4_features = [col for col in model_df_g4.columns if col.startswith("g4_")]
g4_features

['g4_local_60d_mode_genre',
 'g4_local_60d_mode_genre_count',
 'g4_matches_local_60d_mode_genre']

In [56]:
model_df_g4.to_csv(DATA_RAW/"fe_groups/g4.csv")

## Group 5
### Recency Features
- Days since actor’s last movie release.
- Days since director’s last movie release.

In [34]:
def add_group5_recency_features(model_df):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(df["domestic_release_date"], errors="coerce")

    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    entity_cols = {
        "director": "director_id",
        "actor_1": "actor_1",
        "actor_2": "actor_2",
        "actor_3": "actor_3",
        "actor_4": "actor_4",
        "actor_5": "actor_5",
        "actor_6": "actor_6",
    }

    for label, col in entity_cols.items():
        last_release = df.groupby(col)["domestic_release_date"].shift(1)

        df[f"g5_days_since_last_release_{label}"] = (
            df["domestic_release_date"] - last_release
        ).dt.days

        df.loc[df[col].isna(), f"g5_days_since_last_release_{label}"] = np.nan

    actor_recency_cols = [
        "g5_days_since_last_release_actor_1",
        "g5_days_since_last_release_actor_2",
        "g5_days_since_last_release_actor_3",
        "g5_days_since_last_release_actor_4",
        "g5_days_since_last_release_actor_5",
        "g5_days_since_last_release_actor_6",
    ]

    df["g5_top6_actor_days_since_last_release_min"] = df[actor_recency_cols].min(axis=1)
    df["g5_top6_actor_days_since_last_release_mean"] = df[actor_recency_cols].mean(axis=1)
    df["g5_top6_actor_days_since_last_release_max"] = df[actor_recency_cols].max(axis=1)

    return df

In [58]:
model_df_g5 = add_group5_recency_features(model_df)

g5_features = [col for col in model_df_g5.columns if col.startswith("g5_")]
g5_features

['g5_days_since_last_release_director',
 'g5_days_since_last_release_actor_1',
 'g5_days_since_last_release_actor_2',
 'g5_days_since_last_release_actor_3',
 'g5_days_since_last_release_actor_4',
 'g5_days_since_last_release_actor_5',
 'g5_days_since_last_release_actor_6',
 'g5_top6_actor_days_since_last_release_min',
 'g5_top6_actor_days_since_last_release_mean',
 'g5_top6_actor_days_since_last_release_max']

In [59]:
model_df_g5.to_csv(DATA_RAW/"fe_groups/g5.csv")

## Group 6
### Relative Budget Competition Features



In [35]:
def add_group6_relative_budget_features(model_df, window_days=20):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(
        df["domestic_release_date"],
        errors="coerce"
    )

    df["production_budget"] = pd.to_numeric(
        df["production_budget"],
        errors="coerce"
    )

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    local_avg_budget = []
    local_median_budget = []
    local_max_budget = []
    num_larger_budget_competitors = []
    budget_percentile_local = []

    for idx, row in df.iterrows():

        release_date = row["domestic_release_date"]
        budget = row["production_budget"]

        if pd.isna(release_date) or pd.isna(budget):
            local_avg_budget.append(np.nan)
            local_median_budget.append(np.nan)
            local_max_budget.append(np.nan)
            num_larger_budget_competitors.append(np.nan)
            budget_percentile_local.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ]

        nearby_budgets = nearby["production_budget"].dropna()

        if len(nearby_budgets) == 0:
            local_avg_budget.append(np.nan)
            local_median_budget.append(np.nan)
            local_max_budget.append(np.nan)
            num_larger_budget_competitors.append(0)
            budget_percentile_local.append(np.nan)
            continue

        local_avg_budget.append(nearby_budgets.mean())
        local_median_budget.append(nearby_budgets.median())
        local_max_budget.append(nearby_budgets.max())

        num_larger_budget_competitors.append(
            (nearby_budgets > budget).sum()
        )

        percentile = (
            (nearby_budgets < budget).sum()
            / len(nearby_budgets)
        )

        budget_percentile_local.append(percentile)

    df["g6_local_avg_budget_20d"] = local_avg_budget
    df["g6_local_median_budget_20d"] = local_median_budget
    df["g6_local_max_budget_20d"] = local_max_budget

    df["g6_num_larger_budget_competitors_20d"] = (
        num_larger_budget_competitors
    )

    df["g6_budget_percentile_local_20d"] = (
        budget_percentile_local
    )

    # Relative budget ratios
    df["g6_budget_vs_local_avg_ratio"] = (
        df["production_budget"] /
        df["g6_local_avg_budget_20d"]
    )

    df["g6_budget_vs_local_median_ratio"] = (
        df["production_budget"] /
        df["g6_local_median_budget_20d"]
    )

    return df

In [61]:
model_df_g6 = add_group6_relative_budget_features(model_df)

g6_features = [col for col in model_df_g6.columns if col.startswith("g6_")]
g6_features

['g6_local_avg_budget_20d',
 'g6_local_median_budget_20d',
 'g6_local_max_budget_20d',
 'g6_num_larger_budget_competitors_20d',
 'g6_budget_percentile_local_20d',
 'g6_budget_vs_local_avg_ratio',
 'g6_budget_vs_local_median_ratio']

In [62]:
model_df_g6.to_csv(DATA_RAW/"fe_groups/g6.csv")

# Group 8 — Semantic Competition Density

In [24]:
model_df_embedded = pd.read_csv(
    DATA_RAW/"fe_groups/g7_market_synopsis_embeddings.csv"
)

print(model_df_embedded.shape)
model_df_embedded.head()

(2255, 426)


,tconst,primaryTitle,startYear,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,all_domestic_release_types,distributor,production_budget,...,g7_market_synopsis_emb_374,g7_market_synopsis_emb_375,g7_market_synopsis_emb_376,g7_market_synopsis_emb_377,g7_market_synopsis_emb_378,g7_market_synopsis_emb_379,g7_market_synopsis_emb_380,g7_market_synopsis_emb_381,g7_market_synopsis_emb_382,g7_market_synopsis_emb_383
0,tt0120667,Fantastic Four,2005.0,56061504.0,3602.0,2005-07-08,Wide,Wide,20th Century Fox,87500000.0,...,-0.010731,-0.014368,-0.078022,0.003610,-0.058226,-0.032532,-0.038527,-0.072058,-0.074371,-0.039796
1,tt0121164,Corpse Bride,2005.0,19145480.0,3204.0,2005-09-16,Expands Wide,Expands Wide,Warner Bros.,40000000.0,...,0.076011,-0.122633,0.034158,0.081395,0.050829,0.028579,0.025510,0.038926,0.038335,-0.021903
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,108435841.0,3661.0,2005-05-19,Wide,Wide,20th Century Fox,115000000.0,...,-0.062428,0.013874,-0.101026,0.095624,0.053449,0.027899,0.081364,-0.019306,0.028493,-0.003193
3,tt0167190,Hellboy,2004.0,23172440.0,3028.0,2004-04-02,Wide,Wide,Sony Pictures,60000000.0,...,0.031220,-0.089104,0.004114,-0.008470,0.000972,0.000932,0.062636,-0.084109,-0.054190,-0.075343
4,tt0200465,The Bank Job,2008.0,5935256.0,1603.0,2008-03-07,Wide,Wide,Lionsgate,20000000.0,...,0.020103,-0.030943,-0.050717,0.072772,-0.006696,0.006582,-0.032968,-0.043960,0.058457,-0.035927


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

def add_group8_semantic_competition_density(
    model_df,
    window_days=20,
    similarity_threshold=0.70
):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(
        df["domestic_release_date"],
        errors="coerce"
    )

    df = df.sort_values("domestic_release_date").reset_index(drop=True)

    embedding_cols = [
        col for col in df.columns
        if col.startswith("g7_market_synopsis_emb_")
    ]

    if len(embedding_cols) == 0:
        raise ValueError("No g7_market_synopsis_emb_ columns found.")

    embeddings = df[embedding_cols].fillna(0).to_numpy()

    avg_similarities = []
    max_similarities = []
    num_high_similarity = []
    sum_similarity = []

    for idx, row in df.iterrows():
        release_date = row["domestic_release_date"]

        if pd.isna(release_date):
            avg_similarities.append(np.nan)
            max_similarities.append(np.nan)
            num_high_similarity.append(np.nan)
            sum_similarity.append(np.nan)
            continue

        start_date = release_date - pd.Timedelta(days=window_days)
        end_date = release_date + pd.Timedelta(days=window_days)

        nearby_idx = df[
            (df["domestic_release_date"] >= start_date) &
            (df["domestic_release_date"] <= end_date) &
            (df.index != idx)
        ].index.tolist()

        if len(nearby_idx) == 0:
            avg_similarities.append(0)
            max_similarities.append(0)
            num_high_similarity.append(0)
            sum_similarity.append(0)
            continue

        current_emb = embeddings[idx].reshape(1, -1)
        nearby_embs = embeddings[nearby_idx]

        sims = cosine_similarity(current_emb, nearby_embs).flatten()

        avg_similarities.append(sims.mean())
        max_similarities.append(sims.max())
        num_high_similarity.append((sims >= similarity_threshold).sum())
        sum_similarity.append(sims.sum())

    df[f"g8_avg_semantic_similarity_nearby_{window_days}d"] = avg_similarities
    df[f"g8_max_semantic_similarity_nearby_{window_days}d"] = max_similarities
    df[f"g8_num_high_similarity_nearby_{window_days}d"] = num_high_similarity
    df[f"g8_sum_semantic_similarity_nearby_{window_days}d"] = sum_similarity

    return df

In [65]:
model_df_g8 = add_group8_semantic_competition_density(
    model_df_embedded,
    window_days=20,
    similarity_threshold=0.70
)

g8_features = [
    col for col in model_df_g8.columns
    if col.startswith("g8_")
]

g8_features

['g8_avg_semantic_similarity_nearby_20d',
 'g8_max_semantic_similarity_nearby_20d',
 'g8_num_high_similarity_nearby_20d',
 'g8_sum_semantic_similarity_nearby_20d']

In [66]:
model_df_g8.to_csv(
    DATA_RAW / "fe_groups/g8_semantic_competition_density.csv",
    index=False
)

# Group 9 — Dynamic Entity Embeddings

In [37]:
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

def add_group9_dynamic_entity_embeddings(
    model_df,
    entity_cols=None,
    embedding_prefix="g7_market_synopsis_emb_",
    pca_components=25
):
    df = model_df.copy()

    df["domestic_release_date"] = pd.to_datetime(
        df["domestic_release_date"],
        errors="coerce"
    )

    df = df.sort_values(
        ["domestic_release_date", "startYear", "primaryTitle"],
        na_position="last"
    ).reset_index(drop=True)

    if entity_cols is None:
        entity_cols = {
            "director": "director_id",
            "writer": "writer_id",
            "actor_1": "actor_1",
            "actor_2": "actor_2",
            "actor_3": "actor_3",
            "actor_4": "actor_4",
            "actor_5": "actor_5",
            "actor_6": "actor_6",
            "franchise": "franchise"
        }

    embedding_cols = [
        col for col in df.columns
        if col.startswith(embedding_prefix)
    ]

    if len(embedding_cols) == 0:
        raise ValueError(f"No embedding columns found with prefix: {embedding_prefix}")

    X_emb = df[embedding_cols].fillna(0).to_numpy()

    pca = PCA(n_components=pca_components, random_state=42)
    X_pca = pca.fit_transform(X_emb)

    pca_cols = [
        f"g9_movie_synopsis_pca_{i}"
        for i in range(pca_components)
    ]

    pca_df = pd.DataFrame(X_pca, columns=pca_cols, index=df.index)
    df = pd.concat([df, pca_df], axis=1)

    # Store running historical embeddings per entity
    history = {
        label: {}
        for label in entity_cols.keys()
    }

    # Create empty dynamic embedding columns once
    new_cols = {}

    for label, col in entity_cols.items():
        for i in range(pca_components):
            new_cols[f"g9_dynamic_{label}_emb_{i}"] = np.nan

    new_cols_df = pd.DataFrame(
        new_cols,
        index=df.index
    )

    df = pd.concat([df, new_cols_df], axis=1)

    for idx, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Building dynamic embeddings"
    ):
        current_movie_emb = X_pca[idx]

        # First assign prior historical embeddings
        for label, col in entity_cols.items():
            entity_value = row[col]

            if pd.isna(entity_value) or str(entity_value).strip() == "":
                continue

            entity_value = str(entity_value)

            if entity_value in history[label]:
                prior_embeddings = history[label][entity_value]

                prior_mean_embedding = np.mean(
                    prior_embeddings,
                    axis=0
                )

                for i in range(pca_components):
                    df.loc[idx, f"g9_dynamic_{label}_emb_{i}"] = prior_mean_embedding[i]

        # Then update history AFTER current movie
        # This prevents leakage.
        for label, col in entity_cols.items():
            entity_value = row[col]

            if pd.isna(entity_value) or str(entity_value).strip() == "":
                continue

            entity_value = str(entity_value)

            if entity_value not in history[label]:
                history[label][entity_value] = []

            history[label][entity_value].append(current_movie_emb)

    return df

In [35]:
model_df_g9 = add_group9_dynamic_entity_embeddings(
    model_df_embedded,
    pca_components=25
)

g9_features = [
    col for col in model_df_g9.columns
    if col.startswith("g9_dynamic_")
]

len(g9_features), g9_features[:10]

Building dynamic embeddings:   0%|          | 0/2255 [00:00<?, ?it/s]

(225,
 ['g9_dynamic_director_emb_0',
  'g9_dynamic_director_emb_1',
  'g9_dynamic_director_emb_2',
  'g9_dynamic_director_emb_3',
  'g9_dynamic_director_emb_4',
  'g9_dynamic_director_emb_5',
  'g9_dynamic_director_emb_6',
  'g9_dynamic_director_emb_7',
  'g9_dynamic_director_emb_8',
  'g9_dynamic_director_emb_9'])

In [36]:
model_df_g9.to_csv(
    DATA_RAW / "fe_groups/g9_dynamic_entity_embeddings.csv",
    index=False
)

Testing Combos

In [25]:
from itertools import combinations


from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor


In [26]:
def add_all_feature_groups(model_df):
    df = model_df.copy()

    df = add_group1_prior_genre_experience(df)
    df = model_df_g2b = add_group2b_time_aware_momentum(df)
    df = add_group3_release_timing_competition(df)
    df = add_group4_local_genre_environment(df)
    df = add_group5_recency_features(df)
    df = add_group6_relative_budget_features(df)
    df = add_group8_semantic_competition_density(df)
    df =add_group9_dynamic_entity_embeddings(df)

    return df

In [27]:
def get_group_features(df):
    return {
        "g1": [c for c in df.columns if c.startswith("g1_")],
        "g2": [c for c in df.columns if c.startswith("g2_")],
        "g3": [c for c in df.columns if c.startswith("g3_")],
        "g4": [c for c in df.columns if c.startswith("g4_")],
        "g5": [c for c in df.columns if c.startswith("g5_")],
        "g6": [c for c in df.columns if c.startswith("g6_")],
        "g8": [c for c in df.columns if c.startswith("g8_")],
        "g9": [c for c in df.columns if c.startswith("g9_")],

    }

In [28]:
def evaluate_feature_group_combinations(df, base_features, target_col):
    group_features = get_group_features(df)
    groups = list(group_features.keys())

    results = []

    for r in range(1, len(groups) + 1):
        for combo in combinations(groups, r):

            combo_features = []
            for group in combo:
                combo_features.extend(group_features[group])

            selected_features = base_features + combo_features

            test_df = df[selected_features + [target_col]].copy()
            test_df = test_df.dropna(subset=[target_col])

            X = test_df[selected_features]
            y = test_df[target_col]

            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42
            )

            numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
            categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

            numeric_transformer = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ])

            categorical_transformer = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ])

            preprocessor = ColumnTransformer(
                transformers=[
                    ("num", numeric_transformer, numeric_features),
                    ("cat", categorical_transformer, categorical_features)
                ]
            )

            models = {
                "Ridge": Ridge(alpha=1.0),
                "XGBoost": XGBRegressor(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42,
                    objective="reg:squarederror",
                    tree_method="hist",
                    device="cuda"
                )
            }

            for model_name, model in models.items():
                pipe = Pipeline([
                    ("preprocessor", preprocessor),
                    ("model", model)
                ])

                pipe.fit(X_train, y_train)
                preds_log = pipe.predict(X_test)

                r2_log = r2_score(y_test, preds_log)
                mae_log = mean_absolute_error(y_test, preds_log)

                preds_dollars = np.expm1(preds_log)
                actual_dollars = np.expm1(y_test)

                mae_dollars = mean_absolute_error(actual_dollars, preds_dollars)

                results.append({
                    "feature_groups": "+".join(combo),
                    "num_feature_groups": len(combo),
                    "num_engineered_features": len(combo_features),
                    "model": model_name,
                    "r2_log": r2_log,
                    "mae_log": mae_log,
                    "mae_dollars": mae_dollars
                })

    return pd.DataFrame(results).sort_values(
        ["model", "r2_log"],
        ascending=[True, False]
    )

In [39]:
model_df_embedded = model_df_embedded[
    model_df_embedded["franchise"].isna() |
    (model_df_embedded["franchise"].astype(str).str.strip() == "")
].copy()

In [40]:
model_df_all = add_all_feature_groups(model_df_embedded)

base_features = [
    "production_budget",
    "runtime_minutes",
    "release_day_of_year",
    "MPA_rating",
    "genre",
    "subgenre",
    "target_audience",
    "creative_type",
    "production_countries",
    "distributor",
    "source",
    "director_id",
    "writer_id",
    "actor_1",
    "actor_2",
    "actor_3"
]

target_col = "log_opening_weekend_gross"

results_df = evaluate_feature_group_combinations(
    model_df_all,
    base_features=base_features,
    target_col=target_col
)

results_df.head(20)

Building dynamic embeddings:   0%|          | 0/1493 [00:00<?, ?it/s]

c:\Users\sebas\miniconda3\envs\rr\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:38:05] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\sebas\miniconda3\envs\rr\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:38:06] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\sebas\miniconda3\envs\rr\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:38:07] WARNING: D:\bld\xgboost-split_1772124962567\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\sebas\miniconda3\envs\rr\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:38:

,feature_groups,num_feature_groups,num_engineered_features,model,r2_log,mae_log,mae_dollars
158,g3+g6+g8,3,18,Ridge,0.527370,0.866649,8.099545e+06
268,g2+g3+g6+g8,4,18,Ridge,0.527370,0.866649,8.099545e+06
300,g3+g4+g6+g8,4,21,Ridge,0.525006,0.874109,8.177203e+06
400,g2+g3+g4+g6+g8,5,21,Ridge,0.525006,0.874109,8.177203e+06
46,g3+g6,2,14,Ridge,0.524201,0.870969,7.946165e+06
118,g2+g3+g6,3,14,Ridge,0.524201,0.870969,7.946165e+06
228,g1+g3+g6+g8,4,27,Ridge,0.523277,0.872285,8.407806e+06
338,g1+g2+g3+g6+g8,5,27,Ridge,0.523277,0.872285,8.407806e+06
370,g1+g3+g4+g6+g8,5,30,Ridge,0.523226,0.877499,8.437988e+06
442,g1+g2+g3+g4+g6+g8,6,30,Ridge,0.523226,0.877499,8.437988e+06


In [41]:
results_df[results_df["model"] == "XGBoost"].head(15)

,feature_groups,num_feature_groups,num_engineered_features,model,r2_log,mae_log,mae_dollars
229,g1+g3+g6+g8,4,27,XGBoost,0.571015,0.802505,6.089006e+06
339,g1+g2+g3+g6+g8,5,27,XGBoost,0.571015,0.802505,6.089006e+06
97,g1+g4+g6,3,19,XGBoost,0.570854,0.806405,6.291625e+06
197,g1+g2+g4+g6,4,19,XGBoost,0.570854,0.806405,6.291625e+06
315,g4+g5+g6+g8,4,24,XGBoost,0.566784,0.809369,6.264723e+06
415,g2+g4+g5+g6+g8,5,24,XGBoost,0.566784,0.809369,6.264723e+06
385,g1+g4+g5+g6+g8,5,33,XGBoost,0.565341,0.811254,6.374554e+06
457,g1+g2+g4+g5+g6+g8,6,33,XGBoost,0.565341,0.811254,6.374554e+06
241,g1+g4+g6+g8,4,23,XGBoost,0.564645,0.804048,6.212881e+06
351,g1+g2+g4+g6+g8,5,23,XGBoost,0.564645,0.804048,6.212881e+06


In [42]:
results_df[results_df["model"] == "Ridge"].head(15)

,feature_groups,num_feature_groups,num_engineered_features,model,r2_log,mae_log,mae_dollars
158,g3+g6+g8,3,18,Ridge,0.527370,0.866649,8.099545e+06
268,g2+g3+g6+g8,4,18,Ridge,0.527370,0.866649,8.099545e+06
300,g3+g4+g6+g8,4,21,Ridge,0.525006,0.874109,8.177203e+06
400,g2+g3+g4+g6+g8,5,21,Ridge,0.525006,0.874109,8.177203e+06
46,g3+g6,2,14,Ridge,0.524201,0.870969,7.946165e+06
118,g2+g3+g6,3,14,Ridge,0.524201,0.870969,7.946165e+06
228,g1+g3+g6+g8,4,27,Ridge,0.523277,0.872285,8.407806e+06
338,g1+g2+g3+g6+g8,5,27,Ridge,0.523277,0.872285,8.407806e+06
370,g1+g3+g4+g6+g8,5,30,Ridge,0.523226,0.877499,8.437988e+06
442,g1+g2+g3+g4+g6+g8,6,30,Ridge,0.523226,0.877499,8.437988e+06


In [43]:
model_df_all.to_csv(DATA_RAW / "fe_groups/all_feature_groups.csv", index=False)

results_df.to_csv(DATA_PROCESSED/"feature_testing/results/all_group_combination_results.csv", index=False)